# Preprocesado e integración — STARWARS_AUTOCALLS

Notebook que transforma las variables y construye el dataset con el que se entrenan los modelos.

El flujo va de las tres tablas de origen a una única tabla con **una fila por RFQ**:

| Bloque | Qué hace |
|---|---|
| 0 | Carga y validación de las tres fuentes |
| 1 | Señal de tendencia de la volatilidad de mercado |
| 2 | Features disponibles en el momento de cotizar |
| 3 | El plazo del producto (la variable de primer orden) |
| 4 | Explosión a una fila por RFQ–subyacente y cruce con mercado y referencia |
| 5 | Agrupamiento por RFQ: estadísticos de la cesta |
| 6 | Dummies de categóricas y de presencia de tickers |
| 7 | Conjunto de entrenamiento y contrato de features |
| 8 | Guardado de los ficheros procesados |

## 0. Setup y carga de las tres tablas

Cargamos `rfqs`, `daily_volatility` y `underlyings_reference` por separado y validamos las claves
antes de tocar nada: si el grano no es el que creemos, cualquier cruce posterior estaría mal.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# The notebook can run from notebooks/ or from the project root.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "data" / "raw").exists() else cwd.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

rfqs = pd.read_csv(RAW_DIR / "rfqs.csv", parse_dates=["requested_date", "start_date", "end_date"])
vol = pd.read_csv(RAW_DIR / "daily_volatility.csv", parse_dates=["date"])
reference = pd.read_csv(RAW_DIR / "underlyings_reference.csv")

assert rfqs["rfq_id"].is_unique, "rfq_id must be unique"
assert reference["underlying"].is_unique, "Reference tickers must be unique"
assert not vol.duplicated(["date", "underlying"]).any(), "Duplicate (date, ticker) market records"

print(f"rfqs.csv                  : {rfqs.shape[0]:>7,} filas x {rfqs.shape[1]:>2} columnas")
print(f"daily_volatility.csv      : {vol.shape[0]:>7,} filas x {vol.shape[1]:>2} columnas")
print(f"underlyings_reference.csv : {reference.shape[0]:>7,} filas x {reference.shape[1]:>2} columnas")
print("Claves OK: rfq_id único, ticker único en referencia, sin duplicados (date, underlying)")

## 1. Señal de tendencia de la volatilidad

Para cada ticker comparamos su volatilidad realizada de hoy con la media de sus **21 observaciones
anteriores**: positivo = la vol está por encima de su nivel reciente, negativo = por debajo.
El `shift(1)` es importante — la media de referencia no incluye el día que estamos mirando.

In [ ]:
# Trend of the volatility indicator at each market date. It compares the latest
# 63-day realized volatility with the 21 prior published observations for the same ticker.
# A positive value means volatility is above its recent level; negative means below it.
vol = vol.sort_values(["underlying", "date"]).copy()
vol["realized_vol_21d_prior_mean"] = (
    vol.groupby("underlying")["realized_vol_63d"]
    .transform(lambda s: s.shift(1).rolling(21, min_periods=21).mean())
)
vol["realized_vol_trend_21d"] = (
    vol["realized_vol_63d"] - vol["realized_vol_21d_prior_mean"]
)

sin_tendencia = vol["realized_vol_trend_21d"].isna().sum()
print(f"Observaciones sin tendencia (primeros 21 días de cada ticker): {sin_tendencia:,}")
print()
print("Distribución de realized_vol_trend_21d:")
print(vol["realized_vol_trend_21d"].describe().round(4))

## 2. Features disponibles en el momento de cotizar

Solo usamos información que la mesa tiene **cuando llega la RFQ**. Normalizamos la frecuencia de
observación a meses (el EDA mostró que venía escrita de varias formas), y sacamos tamaño de cesta,
nominal en log (su distribución es muy asimétrica) y la estacionalidad de la fecha en seno/coseno.

In [ ]:
FREQUENCY_TO_MONTHS = {
    "1d": 1 / 30.44,
    "1m": 1, "m": 1, "monthly": 1, "mensual": 1, "1 month": 1,
    "2m": 2,
    "3m": 3, "q": 3, "quarterly": 3, "trimestral": 3, "3 months": 3,
    "6m": 6,
    "1y": 12, "y": 12, "12m": 12, "annual": 12, "anual": 12,
}
def frequency_to_months(s):
    key = s.astype("string").str.strip().str.lower()
    result = key.map(FREQUENCY_TO_MONTHS)
    unknown = sorted(key[result.isna()].dropna().unique())
    if unknown:
        raise ValueError(f"Unmapped frequencies: {unknown}")
    return result.astype(float)

base = rfqs.copy()
base["observation_frequency_months"] = frequency_to_months(base["observation_frequency"])
base["basket_size"] = base["underlyings"].str.split("|").str.len()
base["log_notional_credits"] = np.log1p(base["notional_credits"])
base["requested_month_sin"] = np.sin(2 * np.pi * base["requested_date"].dt.month / 12)
base["requested_month_cos"] = np.cos(2 * np.pi * base["requested_date"].dt.month / 12)
base["requested_dayofweek"] = base["requested_date"].dt.dayofweek

print("Frecuencias tal como vienen -> meses:")
print(
    base.groupby("observation_frequency")["observation_frequency_months"]
    .agg(["first", "size"])
    .rename(columns={"first": "meses", "size": "n_rfqs"})
    .sort_values("meses")
)
print()
print(f"Tamaño de cesta: de {base['basket_size'].min()} a {base['basket_size'].max()} subyacentes")
print(f"Rango de requested_date: {base['requested_date'].min():%Y-%m-%d} -> {base['requested_date'].max():%Y-%m-%d}")

## 3. El plazo del producto

`nominal_maturity_months = end_date - start_date` es el **vencimiento nominal** pactado: si el
producto es a 3 años o a 10. Es la variable más importante del problema y merece una justificación
explícita, porque sale de dos columnas de fecha que a primera vista parecen "del futuro".

**Por qué es legítima y no es fuga de información:**

- Una RFQ es una petición sobre un producto concreto: nadie pide precio sin decir el plazo
  (*"cotízame un autocall a 3 años sobre KYBR|TECH"*). El plazo es un término del contrato,
  igual que la barrera o la frecuencia de observación.
- El desfase entre `requested_date` y `start_date` es de **0 a 5 días**: el vencimiento se fija
  prácticamente en el mismo acto de la cotización.
- `no_call_period_months <= vencimiento` se cumple en el **100%** de las filas. Los términos del
  contrato ya son coherentes con un plazo que existía al cotizar.
- Que `end_date` solo esté rellena cuando `executed = True` es un artefacto de **cómo el
  front-office registra el booking**, no evidencia de que el plazo fuese desconocido. "No está en
  el CSV" y "no se sabía" son cosas distintas.

**Lo que sí cuesta, y hay que decirlo:** con el plazo dentro no se pueden puntuar las RFQ no
ejecutadas, porque para ellas el CSV no lo trae. En producción esto no es un problema — el plazo
viene en la petición del cliente y la API lo recibe como un input más.

El bloque de abajo mide, además, cuánta señal aporta.

In [ ]:
# El plazo del producto. Solo existe donde executed = True (ver la nota de arriba).
base["nominal_maturity_months"] = (base["end_date"] - base["start_date"]).dt.days / 30.44

con_plazo = base["nominal_maturity_months"].notna()
print(f"RFQ con plazo conocido: {con_plazo.sum():,} de {len(base):,} ({con_plazo.mean():.1%})")
print(f"Coincide exactamente con executed=True: {con_plazo.equals(base['executed'])}")
print()
print("Plazo nominal (meses):")
print(base["nominal_maturity_months"].describe().round(2))
print()

# Cuanta senal aporta, medido solo donde hay target.
ejecutadas = base.loc[base["executed"]]
print(f"corr(plazo, duración media) = {ejecutadas['nominal_maturity_months'].corr(ejecutadas['avg_duration_months']):+.3f}")
print()
print("Duración media observada según el plazo del producto:")
print(
    ejecutadas.groupby((ejecutadas["nominal_maturity_months"] / 12).round().astype(int))["avg_duration_months"]
    .agg(n_rfqs="size", duracion_media="mean")
    .round(2)
    .rename_axis("plazo_años")
)

## 4. Explosión a una fila por RFQ–subyacente

Partimos la cesta (`"KYBR|TECH"`) en tickers para poder cruzar con mercado y referencia:

- **Referencia**: cruce exacto por ticker (tabla estática).
- **Mercado**: `merge_asof` hacia atrás con `allow_exact_matches=False`, es decir la última
  volatilidad publicada **estrictamente antes** de la fecha de solicitud. Así evitamos meter en el
  modelo información que la mesa no tenía todavía al cotizar.

In [ ]:
# One temporary row per RFQ-ticker.
# Reference uses an exact join. Market uses the latest record BEFORE requested_date.
rfq_ticker = (
    base[["rfq_id", "requested_date", "underlyings"]]
    .assign(underlying=lambda x: x["underlyings"].str.split("|"))
    .explode("underlying", ignore_index=True)
    .drop(columns="underlyings")
)
rfq_ticker["underlying"] = rfq_ticker["underlying"].str.strip()
rfq_ticker = rfq_ticker.merge(
    reference, on="underlying", how="left", validate="many_to_one", indicator="reference_match"
)
rfq_ticker = pd.merge_asof(
    rfq_ticker.sort_values(["requested_date", "underlying"]),
    vol.sort_values(["date", "underlying"]),
    left_on="requested_date", right_on="date", by="underlying",
    direction="backward", allow_exact_matches=False,
)
rfq_ticker["market_lag_days"] = (rfq_ticker["requested_date"] - rfq_ticker["date"]).dt.days
assert rfq_ticker["reference_match"].eq("both").all(), "Ticker missing from reference"
assert rfq_ticker["realized_vol_63d"].notna().all(), "RFQ-ticker missing prior market history"
assert rfq_ticker["realized_vol_trend_21d"].notna().all(), "RFQ-ticker missing prior trend history"

print(f"Filas RFQ-ticker: {len(rfq_ticker):,} (a partir de {len(base):,} RFQ)")
print(f"Tickers distintos: {rfq_ticker['underlying'].nunique()}")
print("Todos los tickers tienen ficha de referencia y vol anterior a la cotización: OK")
print()
print("Antigüedad del dato de mercado usado (días entre requested_date y la última vol publicada):")
print(rfq_ticker["market_lag_days"].describe().round(2))

## 5. Agrupamiento por RFQ: estadísticos de la cesta

Volvemos a **una fila por RFQ** resumiendo la cesta: min/max/media/dispersión de la vol realizada y
de la estructural. El `min` y el `max` importan porque el autocall lo decide el subyacente **más
débil**, no la media. En las cestas de un solo subyacente la desviación es NaN y la ponemos a 0.

`n_underlyings`, `n_market_matches` y `market_match_rate` se calculan como **control de calidad**
del cruce, no como variables del modelo: las tres son redundantes con `basket_size` y quedan fuera
del contrato en el bloque 7.

In [ ]:
# Aggregate back to exactly one row per RFQ.
basket = rfq_ticker.groupby("rfq_id", as_index=False).agg(
    n_underlyings=("underlying", "size"),
    n_market_matches=("realized_vol_63d", "count"),
    realized_vol_min=("realized_vol_63d", "min"),
    realized_vol_max=("realized_vol_63d", "max"),
    realized_vol_mean=("realized_vol_63d", "mean"),
    realized_vol_std=("realized_vol_63d", "std"),
    realized_vol_trend_21d_mean=("realized_vol_trend_21d", "mean"),
    structural_base_vol_min=("structural_base_vol", "min"),
    structural_base_vol_max=("structural_base_vol", "max"),
    structural_base_vol_mean=("structural_base_vol", "mean"),
    structural_base_vol_std=("structural_base_vol", "std"),
    market_lag_days_max=("market_lag_days", "max"),
)
for col in ["realized_vol_std", "structural_base_vol_std"]:
    basket[col] = basket[col].fillna(0.0)  # Single products have no dispersion.
basket["realized_vol_range"] = basket["realized_vol_max"] - basket["realized_vol_min"]
basket["structural_base_vol_range"] = basket["structural_base_vol_max"] - basket["structural_base_vol_min"]
basket["realized_minus_structural_vol"] = basket["realized_vol_mean"] - basket["structural_base_vol_mean"]
basket["market_match_rate"] = basket["n_market_matches"] / basket["n_underlyings"]
assert basket["rfq_id"].is_unique and basket["market_match_rate"].eq(1).all()

print(f"Cestas agregadas: {len(basket):,} (una fila por RFQ, sin pérdidas)")
print()
print("Estadísticos de cesta:")
print(
    basket[[
        "realized_vol_mean", "realized_vol_range",
        "structural_base_vol_mean", "realized_minus_structural_vol",
    ]].describe().round(3)
)

## 6. Dummies: categóricas del contrato y presencia de tickers

La cesta como texto tiene miles de combinaciones distintas, pero solo hay 14 tickers atómicos: en
vez de un one-hot de combinaciones creamos una dummy `has_underlying_*` por ticker. Las categóricas
del contrato (producto, tipo de cesta, contraparte, trader) van a one-hot normal.

In [ ]:
# Categorical dummy variables and ticker-presence dummy variables.
# Do not one-hot 2,043 full basket combinations: there are only 14 atomic tickers.
ticker_dummies = (
    pd.crosstab(rfq_ticker["rfq_id"], rfq_ticker["underlying"])
    .clip(upper=1).add_prefix("has_underlying_").reset_index()
)
features = base.merge(basket, on="rfq_id", how="left", validate="one_to_one")
features = features.merge(ticker_dummies, on="rfq_id", how="left", validate="one_to_one")
categoricals = ["product_type", "basket_type", "counterparty", "trader_id"]
features[categoricals] = features[categoricals].fillna("UNKNOWN")
features = pd.get_dummies(features, columns=categoricals, prefix=categoricals, dtype="int8")
assert len(features) == len(rfqs) and features["rfq_id"].is_unique

print(f"Combinaciones distintas de cesta : {rfqs['underlyings'].nunique():,}")
print(f"Dummies de ticker creadas        : {ticker_dummies.shape[1] - 1}")
print()
print("Dummies por variable categórica:")
for c in categoricals:
    print(f"  {c:<15}: {sum(col.startswith(c + '_') for col in features.columns)}")
print()
print(f"Tabla integrada: {features.shape[0]:,} filas x {features.shape[1]} columnas")

## 7. Conjunto de entrenamiento y contrato de features

Solo las RFQ **ejecutadas** tienen target, así que el modelo se entrena con ellas; el resto se
guarda igualmente para inferencia. Aquí se fija la lista de columnas del modelo, que es el mismo
contrato que usa la API.

Tres grupos quedan fuera del contrato, cada uno por un motivo distinto:

| Grupo | Por qué queda fuera |
|---|---|
| **No conocidas al cotizar** | identificador, target, resultado de la ejecución y fechas |
| **Redundantes o inútiles** | duplican a otra columna, son constantes, o no se pueden extrapolar |
| **Experimentales** | probadas y descartadas, se conservan en la tabla para futuras pruebas |

El grupo de redundantes es el que más ruido quitaba: `basket_size`, `n_underlyings` y
`n_market_matches` eran **la misma columna repetida tres veces**, lo que repartía su importancia en
el SHAP y hacía parecer poco relevante a una de las variables más predictivas del modelo.

In [ ]:
TARGET = "avg_duration_months"

# 1) No se conocen en el momento de cotizar (o no son variables).
#    Ojo: start_date/end_date salen del modelo, pero su diferencia -el plazo- si entra.
#    Es un termino pactado del contrato, no informacion del futuro (ver bloque 3).
NON_MODEL_COLUMNS = [
    "rfq_id", TARGET, "executed", "underlyings", "observation_frequency",
    "requested_date", "start_date", "end_date",
]

# 2) Redundantes o inutiles. Se calculan igual porque sirven de control de calidad.
REDUNDANT_COLUMNS = {
    "n_underlyings": "idéntica a basket_size",
    "n_market_matches": "idéntica a basket_size (lo garantiza el assert del bloque 5)",
    "market_match_rate": "constante = 1.0 por construcción",
    "notional_credits": "misma información que log_notional_credits",
    "requested_year": "el modelo nunca ve el año del test: no puede extrapolar",
    "market_lag_days_max": "artefacto del calendario de publicación, no del producto",
}

# 3) Probadas y descartadas (ver models/reports/volatility_trend_experiment.md).
EXPERIMENTAL_ONLY_COLUMNS = {
    "realized_vol_trend_21d_mean": "no mejoró el MAE, se conserva para futuras pruebas",
}

all_rfqs_features = features.copy()
train_features = features.loc[features["executed"]].copy()

excluded = set(NON_MODEL_COLUMNS) | set(REDUNDANT_COLUMNS) | set(EXPERIMENTAL_ONLY_COLUMNS)
model_feature_columns = [c for c in train_features.columns if c not in excluded]

X = train_features[model_feature_columns]
y = train_features[TARGET]
assert y.notna().all()
assert X.select_dtypes(include="object").empty
assert not X.isna().any().any(), "Model features contain missing values"
assert "nominal_maturity_months" in model_feature_columns, "El plazo debe estar en el contrato"

print("Fuera del contrato por redundantes o inútiles:")
for col, motivo in REDUNDANT_COLUMNS.items():
    print(f"  - {col:<22} {motivo}")
print()
print("Fuera del contrato por experimentales:")
for col, motivo in EXPERIMENTAL_ONLY_COLUMNS.items():
    print(f"  - {col:<22} {motivo}")
print()
print(f"RFQ totales            : {len(all_rfqs_features):,}")
print(f"RFQ ejecutadas (target): {len(train_features):,} ({len(train_features) / len(all_rfqs_features):.1%})")
print(f"Features del modelo    : {len(model_feature_columns)}")
print()
print(f"Target ({TARGET}): media {y.mean():.2f} meses | mediana {y.median():.2f} | rango {y.min():.2f}-{y.max():.2f}")

## 8. Guardado de los ficheros procesados

Tres ficheros: la tabla completa (para inferencia), la de entrenamiento (con target) y la lista de
features, que es lo que garantiza que entrenamiento y API vean exactamente las mismas columnas.

`model_feature_columns.csv` es la **única fuente de verdad** del contrato: los notebooks de modelos
lo leen en vez de volver a deducir la lista por su cuenta, para que no puedan desincronizarse.

In [ ]:
# Reproducible outputs for training and inference.
all_rfqs_features.to_csv(PROCESSED_DIR / "all_rfqs_features.csv", index=False)
train_features.to_csv(PROCESSED_DIR / "train_features.csv", index=False)
pd.Series(model_feature_columns, name="feature_name").to_csv(
    PROCESSED_DIR / "model_feature_columns.csv", index=False
)

print(f"Guardado en: {PROCESSED_DIR}")
for name, n_rows, n_cols in [
    ("all_rfqs_features.csv", len(all_rfqs_features), all_rfqs_features.shape[1]),
    ("train_features.csv", len(train_features), train_features.shape[1]),
    ("model_feature_columns.csv", len(model_feature_columns), 1),
]:
    print(f"  {name:<26} {n_rows:>7,} filas x {n_cols:>3} columnas")
print()
print("Nota: en all_rfqs_features.csv, las RFQ no ejecutadas tienen nulo el plazo y el target.")

### Outputs

- `data/processed/all_rfqs_features.csv`: tabla integrada con todas las RFQ.
- `data/processed/train_features.csv`: RFQ ejecutadas, incluyendo el target y el plazo.
- `data/processed/model_feature_columns.csv`: contrato de features para entrenamiento e inferencia.